# Outlier analysis — 5-fold cross-validated hard-case detection

400k galaxies, 3 models (LR, RF, MLP), **5-fold CV**. Each fold: train all 3 on the 4 training
folds, predict the held-out fold, take the **intersection of the 3 models' outliers**
(|Δz/(1+z)| > 0.05). Every galaxy is tested exactly once out-of-fold, so the collected hard set is
robust to a single train/test split. Hard objids are written to `hard_outliers_cv.csv`.

In [1]:
# --- load + featurize + subsample to 400k ---
import pandas as pd, numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cat = pd.read_parquet("../catalog_v1.parquet")
num = cat.select_dtypes("number").columns
cat[num] = cat[num].mask(cat[num] <= -100)                         # clean SDSS -9999 sentinel
for a, b in [("u","g"), ("g","r"), ("r","i"), ("i","z")]:
    cat[f"{a}-{b}"] = (cat[f"dered_{a}"] - cat[f"dered_{b}"]).clip(-1, 4)
for s in ["expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r"]:
    cat["log_" + s] = np.log1p(cat[s].clip(lower=0))
cat["conc_r"] = cat["petroR90_r"] / cat["petroR50_r"].replace(0, np.nan)

feats = ["dered_u","dered_g","dered_r","dered_i","dered_z", "g-r","u-g","r-i","i-z",
         "log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r",
         "fracDeV_r","conc_r"]

D = cat[feats + ["redshift", "objid"]].replace([np.inf, -np.inf], np.nan).dropna()
D = D.sample(400_000, random_state=0).reset_index(drop=True)       # 400k subset
print(f"D = {len(D)} rows, {len(feats)} features")

D = 400000 rows, 16 features


In [2]:
# --- Experiment 1 (full): 5-fold CV -> per-model metrics + OOF predictions + 3-model intersection ---
# Heavy: RF + MLP trained 5x on ~320k each (a few minutes). Swap RF -> HistGradientBoostingRegressor
# to go ~5-10x faster. Saves OOF predictions so the next cell can plot Δz distributions.
from sklearn.metrics import mean_absolute_error

def make_models():
    return {
        "LR":  make_pipeline(StandardScaler(), LinearRegression()),
        "RF":  RandomForestRegressor(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=0),
        "MLP": make_pipeline(StandardScaler(),
                   MLPRegressor(hidden_layer_sizes=(128, 64, 32), alpha=1e-4, batch_size=256,
                                early_stopping=True, n_iter_no_change=12, max_iter=300, random_state=0)),
    }
def smad(dz): return 1.4826 * np.median(np.abs(dz - np.median(dz)))

X, y, oid = D[feats], D["redshift"], D["objid"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = pd.DataFrame(np.nan, index=D.index, columns=["LR", "RF", "MLP"])   # out-of-fold predictions
metrics, inter_ids = [], []
for fold, (tr, te) in enumerate(kf.split(X)):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    masks = {}
    for name, m in make_models().items():
        m.fit(Xtr, ytr); pred = m.predict(Xte)
        oof.iloc[te, oof.columns.get_loc(name)] = pred           # store OOF prediction
        dz = (pred - yte.values) / (1 + yte.values)
        masks[name] = np.abs(dz) > 0.05
        metrics.append((fold, name, mean_absolute_error(yte, pred), smad(dz), masks[name].mean()))
    inter = masks["LR"] & masks["RF"] & masks["MLP"]
    inter_ids += list(oid.iloc[te].values[inter])
    print(f"fold {fold}: intersection={int(inter.sum())}  " +
          "  ".join(f"{k}={int(v.sum())}" for k, v in masks.items()))

met = pd.DataFrame(metrics, columns=["fold", "model", "MAE", "sigma_MAD", "outlier"])
hard = pd.DataFrame({"objid": pd.Series(inter_ids, dtype="int64")})
hard.to_csv("hard_outliers_cv.csv", index=False)
print("\nper-model metrics (mean +/- std over folds):")
print(met.groupby("model")[["MAE", "sigma_MAD", "outlier"]].agg(["mean", "std"]).round(4).to_string())
print(f"\nhard set (all-3 OOF outliers): {len(hard)}  ({len(hard)/len(D):.2%})")

/home/hzhang/.pyenv/versions/macrocosm/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


KeyboardInterrupt: 

In [3]:
# --- Δz distribution: hard vs full population, under RF and MLP (from OOF predictions) ---
import matplotlib.pyplot as plt
dz_all = oof.sub(y, axis=0).div(1 + y, axis=0)             # OOF Δz per model, every galaxy
hard_mask = oid.isin(set(hard["objid"]))
print(f"hard redshift median={y[hard_mask].median():.3f}   full={y.median():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, mdl in zip(axes, ["RF", "MLP"]):
    full = dz_all[mdl].dropna()
    hd   = dz_all.loc[hard_mask, mdl].dropna()
    bins = np.linspace(-0.3, 0.3, 90)
    ax.hist(full, bins=bins, density=True, alpha=.5, label=f"all ({len(full):,})")
    ax.hist(hd,   bins=bins, density=True, alpha=.6, label=f"hard ({len(hd):,})")
    ax.axvline(0, color="k", lw=.6)
    for t in (-0.05, 0.05):
        ax.axvline(t, color="r", ls=":", lw=.9)
    ax.set_xlabel(r"$\Delta z=(z_{pred}-z_{true})/(1+z)$")
    ax.set_title(f"{mdl}: Δz distribution  (red dotted = outlier threshold ±0.05)")
    ax.legend()
plt.tight_layout(); plt.show()

# How to read it:
#  - 'all' is a tight spike at 0; 'hard' is heavy-tailed (by construction |Δz|>0.05).
#  - if the hard histogram is skewed NEGATIVE -> faint/high-z hard galaxies are systematically
#    UNDER-predicted (pulled toward the crowded low-z region) = a correctable bias, not just scatter.
#  - compare RF vs MLP: similar hard tails -> the failure is data-driven, not model-specific.

NameError: name 'oof' is not defined